# Advanced Python `__slots__`: a step-by-step problem-solving tutorial

**A second, independent notebook** inspired by the short `Location` demonstration. Instead of presenting one large solution per exercise, each problem unfolds through a prediction, a small observation, a design decision, a worked implementation, and checks.

**Audience:** programmers who already understand classes, properties, inheritance and exceptions. **Python:** 3.10 or newer; only the standard library is required. Work through the cells **in order**; use *Restart Kernel and Run All* to reproduce all examples.

## Learning objectives

By the end you will be able to explain why slots change *instance storage* rather than the class namespace; distinguish an unset slot from a slot storing `None`; design validated and cached slotted objects; reason about subclass dictionaries, multiple inheritance, descriptors, name mangling, weak references and serialization; and measure storage without making misleading performance claims.

Every solution is followed by runnable `assert` checks. Some cells deliberately provoke exceptions, but catch them so that **Run All completes successfully**.

## Before starting: two corrections and a rule of thumb

The original Mumbai example accidentally passed a latitude-like value as longitude and vice versa. In geographical coordinates, **latitude** is conventionally in `[-90, 90]` and **longitude** in `[-180, 180]`. This notebook consistently uses explicit keyword arguments to avoid that mistake.

`__slots__` typically prevents an instance dictionary when **every class in the relevant inheritance chain** cooperates. It does **not** freeze an object, make its members private, or prohibit modifying attributes on the *class*. Memory and speed differences depend on implementation, workload, and what you measure.

In [1]:
import copy
import gc
import math
import pickle
import sys
import timeit
import tracemalloc
import weakref
from dataclasses import dataclass, fields
from types import MemberDescriptorType


def expect_exception(exception_type, action):
    """Assert that calling action raises the expected exception type."""
    try:
        action()
    except exception_type as exc:
        return str(exc)
    except Exception as exc:
        raise AssertionError(
            f"Expected {exception_type.__name__}, got {type(exc).__name__}"
        ) from exc
    raise AssertionError(f"Expected {exception_type.__name__} to be raised")


print(f"Python {sys.version_info.major}.{sys.version_info.minor}; ready for the tutorial")

Python 3.13; ready for the tutorial


---

# Problem 01 — A validated geographical object

**Scenario.** You need millions of coordinate objects, but invalid numbers must not silently enter the system.

**Your task.** Expose `latitude` and `longitude` as checked properties and store values only in two private slots. Reject booleans, NaN and infinity as well as out-of-range coordinates.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Predict

Would declaring `__slots__ = ("latitude", "longitude")` and then defining properties named `latitude` and `longitude` work? **No:** slot names conflict with class variables such as properties of the same name. Reserve different *backing* names for the slots.

In [2]:
class BarePoint:
    __slots__ = ("latitude", "longitude")


bare = BarePoint()
bare.latitude = 19.0760
bare.longitude = 72.8777
print((bare.latitude, bare.longitude))
print("Instance has a dictionary?", hasattr(bare, "__dict__"))
assert not hasattr(bare, "__dict__")

(19.076, 72.8777)
Instance has a dictionary? False


### Step 2 — Design the invariant

`bool` is a subclass of `int`, so an ordinary `isinstance(value, (int, float))` check also accepts `True`. In this exercise, bool is not a coordinate. Convert other real numeric inputs to `float` *after* excluding bool, reject non-finite values, and check the interval. A property routes ordinary assignments through validation.

In [3]:
def checked_coordinate(value, *, name, lower, upper):
    if isinstance(value, bool):
        raise TypeError(f"{name} must not be boolean")
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError) as exc:
        raise TypeError(f"{name} must be a real number") from exc
    if not math.isfinite(number) or not lower <= number <= upper:
        raise ValueError(f"{name} must be finite and within [{lower}, {upper}]")
    return number


print(checked_coordinate(19.076, name="latitude", lower=-90, upper=90))
print(expect_exception(ValueError, lambda: checked_coordinate(float("nan"), name="latitude", lower=-90, upper=90)))

19.076
latitude must be finite and within [-90, 90]


### Step 3 — Worked solution

The public properties never share names with the slot descriptors. `__repr__` accesses the validated public API; a caller can assign new valid values after construction.

In [4]:
class GeoPoint:
    __slots__ = ("_latitude", "_longitude")

    def __init__(self, *, latitude, longitude):
        self.latitude = latitude
        self.longitude = longitude

    @property
    def latitude(self):
        return self._latitude

    @latitude.setter
    def latitude(self, value):
        self._latitude = checked_coordinate(
            value, name="latitude", lower=-90, upper=90
        )

    @property
    def longitude(self):
        return self._longitude

    @longitude.setter
    def longitude(self, value):
        self._longitude = checked_coordinate(
            value, name="longitude", lower=-180, upper=180
        )

    def __repr__(self):
        return f"GeoPoint(latitude={self.latitude!r}, longitude={self.longitude!r})"


mumbai = GeoPoint(latitude=19.0760, longitude=72.8777)
print(mumbai)
mumbai.latitude = 19.1
assert mumbai.latitude == 19.1
assert mumbai.longitude == 72.8777
assert not hasattr(mumbai, "__dict__")

GeoPoint(latitude=19.076, longitude=72.8777)


### Step 4 — Adversarial checks

The `try`/`except` helper is deliberate: a notebook containing raw expected exceptions would stop at the first failed line. Notice that a writable *backing slot* remains accessible to a determined caller: slots provide storage control, not security or perfect immutability.

In [5]:
for bad_latitude in (91, -91, float("inf"), float("nan")):
    expect_exception(ValueError, lambda x=bad_latitude: setattr(mumbai, "latitude", x))
for bad_type in (True, None, "not-a-number"):
    expect_exception(TypeError, lambda x=bad_type: setattr(mumbai, "latitude", x))
expect_exception(AttributeError, lambda: setattr(mumbai, "description", "city"))
assert mumbai.latitude == 19.1  # Rejected writes leave the old value unchanged.
print("All coordinate invariants and the no-dynamic-attribute rule passed.")

All coordinate invariants and the no-dynamic-attribute rule passed.


**Takeaway.** A slot restricts *where* instance values live; a property determines *how* an ordinary public assignment is validated. Neither mechanism alone provides the other's guarantees.

---

# Problem 02 — Unset, `None`, deleted: three different states

**Scenario.** A configuration object contains an optional note. A missing note, a note deliberately set to `None`, and a deleted note should be distinguishable.

**Your task.** Inspect raw slot behavior; then implement a clear API that tells these states apart without relying on an instance dictionary.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Small experiment

Slots start *unassigned* if `__init__` does not fill them. Access raises `AttributeError`. Unlike a mapping, a slot has no key that can be enumerated from an instance dictionary.

In [6]:
class RawState:
    __slots__ = ("note",)


raw_state = RawState()
print("Initially present?", hasattr(raw_state, "note"))
expect_exception(AttributeError, lambda: raw_state.note)
raw_state.note = None
assert hasattr(raw_state, "note") and raw_state.note is None
del raw_state.note
assert not hasattr(raw_state, "note")
raw_state.note = "restored"
assert raw_state.note == "restored"

Initially present? False


### Step 2 — Choose semantics

For an optional field, using a dedicated sentinel avoids conflating an unset value with `None`. Here `UNSET` means “never supplied or explicitly cleared.” We also demonstrate that deleting a public property can be *controlled* by a deleter rather than deleting its backing slot.

In [7]:
UNSET = object()


class NoteRecord:
    __slots__ = ("identifier", "_note")

    def __init__(self, identifier, note=UNSET):
        self.identifier = identifier
        self._note = note

    @property
    def has_note(self):
        return self._note is not UNSET

    @property
    def note(self):
        if not self.has_note:
            raise AttributeError("note has not been supplied")
        return self._note

    @note.setter
    def note(self, value):
        self._note = value

    @note.deleter
    def note(self):
        self._note = UNSET


record = NoteRecord("R-001")
assert not record.has_note
expect_exception(AttributeError, lambda: record.note)
record.note = None
assert record.has_note and record.note is None
del record.note
assert not record.has_note
record.note = "reviewed"
assert record.note == "reviewed"
print("Missing -> None -> missing -> string: all distinct transitions passed.")

Missing -> None -> missing -> string: all distinct transitions passed.


### Step 3 — Edge-case analysis

A user can still assign `_note` directly, and can delete it too; ordinary single-underscore names are a convention. For high-reliability domain objects, keep invariants in a documented API and do not claim that `__slots__` enforces privacy.

In [8]:
record_two = NoteRecord("R-002", note=None)
assert record_two.has_note
expect_exception(AttributeError, lambda: setattr(record_two, "extra", 1))
assert type(NoteRecord.__dict__) is type(GeoPoint.__dict__)
print("The class has a mapping proxy; its instances need not have a dictionary.")

The class has a mapping proxy; its instances need not have a dictionary.


**Takeaway.** Uninitialized storage and a stored `None` are not equivalent. A sentinel plus a property gives precise state semantics.

---

# Problem 03 — Slot descriptors and the class namespace

**Scenario.** A teammate sees `__dict__` on a slotted class and assumes its instances have dictionaries too.

**Your task.** Inspect the different objects involved, add a class attribute, and safely demonstrate why overwriting a slot descriptor on a class is hazardous.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — The class still has a namespace

`SomeClass.__dict__` is a read-only **view of the class namespace** (`mappingproxy`). Slot declarations create descriptors inside that namespace. It is the *instance* dictionary that is usually omitted, not the class dictionary.

In [9]:
class DescriptorDemo:
    __slots__ = ("value",)


demo = DescriptorDemo()
demo.value = 42
slot_descriptor = DescriptorDemo.__dict__["value"]
print("Class namespace type:", type(DescriptorDemo.__dict__).__name__)
print("Storage descriptor type:", type(slot_descriptor).__name__)
print("Read via descriptor:", slot_descriptor.__get__(demo, DescriptorDemo))
assert isinstance(slot_descriptor, MemberDescriptorType)
assert not hasattr(demo, "__dict__")

Class namespace type: mappingproxy
Storage descriptor type: member_descriptor
Read via descriptor: 42


### Step 2 — A safe class-level change

Adding a *different* class attribute changes `DescriptorDemo.__dict__`, and every instance can resolve the name through normal attribute lookup. It does not add per-instance storage.

In [10]:
DescriptorDemo.category = "demonstration"
assert demo.category == "demonstration"
assert "category" in DescriptorDemo.__dict__
expect_exception(AttributeError, lambda: setattr(demo, "category", "local"))
del DescriptorDemo.category
assert "category" not in DescriptorDemo.__dict__

### Step 3 — Why overwriting a slot descriptor is unsafe

We use a disposable class and restore its descriptor with `try/finally`. Replacing a slot descriptor with a plain class attribute hides normal access to the stored value; it does **not** reconfigure instance memory.

In [11]:
class TemporarySlot:
    __slots__ = ("amount",)


temporary = TemporarySlot()
temporary.amount = 100
saved_descriptor = TemporarySlot.__dict__["amount"]
try:
    TemporarySlot.amount = "shadowed at class level"
    assert temporary.amount == "shadowed at class level"
    expect_exception(AttributeError, lambda: setattr(temporary, "amount", 200))
finally:
    TemporarySlot.amount = saved_descriptor
assert temporary.amount == 100
print("Original stored value was recovered after descriptor restoration.")

Original stored value was recovered after descriptor restoration.


**Takeaway.** Class attributes are mutable independently of instance layout. Avoid redefining slot descriptors or reusing their names for class-level constants.

---

# Problem 04 — Inheritance: where did the instance dictionary come from?

**Scenario.** You inherit from a compact base class and unexpectedly find that derived objects accept arbitrary attributes.

**Your task.** Build three subclasses, determine which ones have instance dictionaries, and explain why an inherited dictionary cannot be undone merely by introducing slots later.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Compare three subclasses

An otherwise ordinary subclass of a slotted class gets `__dict__` (and generally `__weakref__`) unless it declares its own slots. `__slots__ = ()` means “no additional instance storage” for a subclass.

In [12]:
class SlottedBase:
    __slots__ = ("identifier",)

    def __init__(self, identifier):
        self.identifier = identifier


class AccidentalDict(SlottedBase):
    pass


class NoNewFields(SlottedBase):
    __slots__ = ()


class ExtraSlottedField(SlottedBase):
    __slots__ = ("priority",)


a = AccidentalDict("A")
b = NoNewFields("B")
c = ExtraSlottedField("C")
a.unplanned = "allowed"
c.priority = 5
assert a.__dict__ == {"unplanned": "allowed"}
assert not hasattr(b, "__dict__")
assert not hasattr(c, "__dict__")
expect_exception(AttributeError, lambda: setattr(b, "unplanned", "rejected"))
expect_exception(AttributeError, lambda: setattr(c, "unplanned", "rejected"))
print("base child:", a.__dict__, " | empty-slots child:", b.identifier, " | new-slot child:", c.priority)

base child: {'unplanned': 'allowed'}  | empty-slots child: B  | new-slot child: 5


### Step 2 — The opposite inheritance direction

A slotted subclass does not remove a dictionary inherited from a non-slotted base. This is a common source of misleading memory comparisons: declaring `__slots__` somewhere in the hierarchy is not sufficient.

In [13]:
class FlexibleBase:
    def __init__(self):
        self.base_value = 10


class StillFlexible(FlexibleBase):
    __slots__ = ("own_slot",)


flexible = StillFlexible()
flexible.own_slot = 20
flexible.surprise = 30
assert flexible.__dict__ == {"base_value": 10, "surprise": 30}
assert flexible.own_slot == 20
print("Inherited dictionary survives:", flexible.__dict__)

Inherited dictionary survives: {'base_value': 10, 'surprise': 30}


### Step 3 — Design rule

Audit the **entire method-resolution order** before assuming objects have no dictionary. In an API intended for extension, document whether derived classes should declare `__slots__ = ()` when they add no storage.

In [14]:
for cls in (AccidentalDict, NoNewFields, ExtraSlottedField, StillFlexible):
    print(cls.__name__, "->", " -> ".join(item.__name__ for item in cls.__mro__))
assert "__dict__" in AccidentalDict.__dict__
assert "__dict__" not in NoNewFields.__dict__

AccidentalDict -> AccidentalDict -> SlottedBase -> object
NoNewFields -> NoNewFields -> SlottedBase -> object
ExtraSlottedField -> ExtraSlottedField -> SlottedBase -> object
StillFlexible -> StillFlexible -> FlexibleBase -> object


**Takeaway.** Instance layout is an inheritance-chain property, not an isolated class declaration.

---

# Problem 05 — Multiple inheritance without layout conflicts

**Scenario.** You want to add logging-like behavior to slotted records through a reusable mixin.

**Your task.** First see why two unrelated bases with populated slot layouts cannot generally be combined. Then implement a stateless mixin using empty slots.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — A deliberately invalid design

For ordinary Python classes, two independent bases with incompatible nonempty slot layouts commonly trigger `TypeError: multiple bases have instance lay-out conflict`. Catch the error, rather than leaving the notebook in a failed state.

In [15]:
class HasX:
    __slots__ = ("x",)


class HasY:
    __slots__ = ("y",)


def create_conflicting_class():
    class Conflicted(HasX, HasY):
        __slots__ = ()
    return Conflicted


message = expect_exception(TypeError, create_conflicting_class)
print("Expected layout error:", message)

Expected layout error: multiple bases have instance lay-out conflict


### Step 2 — A mixin that does not require instance storage

A mixin often only supplies methods. Use `__slots__ = ()` to avoid adding a dictionary and avoid claiming a second independent object layout.

In [16]:
class DescriptiveMixin:
    __slots__ = ()

    def describe(self):
        return f"{type(self).__name__}(identifier={self.identifier!r})"


class DescribedItem(DescriptiveMixin, SlottedBase):
    __slots__ = ("quantity",)

    def __init__(self, identifier, quantity):
        super().__init__(identifier)
        self.quantity = quantity


item = DescribedItem("part-7", 12)
print(item.describe())
assert item.identifier == "part-7" and item.quantity == 12
assert not hasattr(item, "__dict__")
expect_exception(AttributeError, lambda: setattr(item, "unknown", "oops"))

DescribedItem(identifier='part-7')


"'DescribedItem' object has no attribute 'unknown' and no __dict__ for setting new attributes"

### Step 3 — Inspect the reason it works

The stateless mixin provides behavior, while `SlottedBase` contributes the original storage layout and `DescribedItem` adds one slot. Real multiple inheritance can be more complex (especially extension types); test the exact hierarchy you intend to support.

In [17]:
print("MRO:", " -> ".join(cls.__name__ for cls in DescribedItem.__mro__))
assert DescriptiveMixin.__slots__ == ()
assert DescribedItem.__slots__ == ("quantity",)

MRO: DescribedItem -> DescriptiveMixin -> SlottedBase -> object


**Takeaway.** Put behavior-only mixins on an empty layout and leave required instance storage to one compatible inheritance branch.

---

# Problem 06 — Name mangling is not access control

**Scenario.** An internal token must avoid collisions with names used by subclasses.

**Your task.** Use a double-underscore slot, inspect its real name, and create a subclass with its own double-underscore slot without collision.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Predict the stored name

Within class `TokenOwner`, an identifier like `__token` undergoes *name mangling* to `_TokenOwner__token`. This helps avoid accidental clashes in subclasses. It is not secrecy, encryption, or access control.

In [18]:
class TokenOwner:
    __slots__ = ("__token",)

    def __init__(self, token):
        self.__token = token

    def own_token(self):
        return self.__token


owner = TokenOwner("base-secret")
print("Declaring class slots:", TokenOwner.__slots__)
print("Actual descriptor:", [key for key in TokenOwner.__dict__ if "token" in key])
assert owner.own_token() == "base-secret"
assert owner._TokenOwner__token == "base-secret"
expect_exception(AttributeError, lambda: getattr(owner, "__token"))

Declaring class slots: ('__token',)
Actual descriptor: ['own_token', '_TokenOwner__token']


"'TokenOwner' object has no attribute '__token'"

### Step 2 — A subclass with the same source spelling

Both classes can write `__token` in their own method bodies, yet Python mangles each to a distinct storage name. Explicitly declare the child slot to keep the layout dictionary-free.

In [19]:
class ChildToken(TokenOwner):
    __slots__ = ("__token",)

    def __init__(self, base_token, child_token):
        super().__init__(base_token)
        self.__token = child_token

    def child_token(self):
        return self.__token


child_token = ChildToken("B", "C")
assert child_token.own_token() == "B"
assert child_token.child_token() == "C"
assert child_token._TokenOwner__token == "B"
assert child_token._ChildToken__token == "C"
assert not hasattr(child_token, "__dict__")
print("Two independent slots:", child_token.own_token(), child_token.child_token())

Two independent slots: B C


**Takeaway.** Name mangling is primarily a defense against *accidental name collisions*. Do not store secrets in plain attributes and assume they are protected.

---

# Problem 07 — Weak references, observers, and object lifetime

**Scenario.** A registry must associate observations with objects without keeping every observed object alive forever.

**Your task.** Find out why a basic slotted class is not weak-referenceable, then build an opt-in weak-referenceable class and a `WeakKeyDictionary` cache.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Observe the failure

A plain slotted class normally does not support weak references unless its layout includes `__weakref__` (or an ancestor provides support). `weakref.ref` will raise `TypeError`.

In [20]:
class NotWeakrefable:
    __slots__ = ("key",)


not_weakrefable = NotWeakrefable()
not_weakrefable.key = "N"
print(expect_exception(TypeError, lambda: weakref.ref(not_weakrefable)))

cannot create weak reference to 'NotWeakrefable' object


### Step 2 — Add the necessary slot once

Do not repeat `__weakref__` in a subclass whose base already provides it. A weak-key cache also needs keys to be hashable; leaving identity-based `__eq__` and `__hash__` alone is enough for this example.

In [21]:
class TrackedItem:
    __slots__ = ("identifier", "__weakref__")

    def __init__(self, identifier):
        self.identifier = identifier


observations = weakref.WeakKeyDictionary()
tracked = TrackedItem("TEMP")
weak_handle = weakref.ref(tracked)
observations[tracked] = {"reads": 1}  # Value does not reference tracked.
assert weak_handle() is tracked
assert len(observations) == 1
assert not hasattr(tracked, "__dict__")
print("Cache before object deletion:", len(observations))

Cache before object deletion: 1


### Step 3 — Test lifetime rather than guessing

Remove the last strong reference and request a collection. The weak reference should then become `None`, and accessing the weak-key mapping causes its dead entry to be discarded. On normal CPython this is prompt; explicit `gc.collect()` makes the intent clearer.

In [22]:
del tracked
gc.collect()
assert weak_handle() is None
assert len(observations) == 0
print("Cache after object deletion:", len(observations))

Cache after object deletion: 0


**Takeaway.** Weak-reference support is a layout decision. A weak-key cache is only truly weak if its values do **not** hold strong references back to their keys.

---

# Problem 08 — Descriptor-based validation with slot-backed storage

**Scenario.** Your application has several numeric fields with similar constraints; copy-pasting a property per field is tedious.

**Your task.** Build a reusable data descriptor that stores in an explicitly declared underscored slot and validates on every ordinary assignment.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Separate descriptor from storage

A data descriptor implements `__get__` and `__set__`. For this tutorial, `__set_name__` learns its public name and chooses an underscored backing name. The owning class must actually declare that backing name in `__slots__`. Never put a descriptor and a slot under the *same* class attribute name.

In [23]:
class BoundedNumber:
    def __init__(self, minimum, maximum):
        if minimum > maximum:
            raise ValueError("minimum must not exceed maximum")
        self.minimum = minimum
        self.maximum = maximum
        self.public_name = None
        self.storage_name = None

    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{self.public_name} must be an int or float, not bool")
        numeric = float(value)
        if not math.isfinite(numeric) or not self.minimum <= numeric <= self.maximum:
            raise ValueError(
                f"{self.public_name} must be finite and in "
                f"[{self.minimum}, {self.maximum}]"
            )
        setattr(instance, self.storage_name, numeric)

    def __delete__(self, instance):
        raise AttributeError(f"Cannot delete {self.public_name}")


print("Descriptor class ready; the next cell provides its storage slots.")

Descriptor class ready; the next cell provides its storage slots.


### Step 2 — Compose a validated record

Two *descriptor instances* can coexist safely because each learns a different backing name. The properties are not duplicated, and instances still have no `__dict__`.

In [24]:
class WeatherReading:
    __slots__ = ("station", "_humidity", "_temperature")

    humidity = BoundedNumber(0, 100)
    temperature = BoundedNumber(-100, 80)

    def __init__(self, station, *, humidity, temperature):
        self.station = station
        self.humidity = humidity
        self.temperature = temperature


reading = WeatherReading("north", humidity=40, temperature=22.5)
assert (reading.humidity, reading.temperature) == (40.0, 22.5)
assert isinstance(WeatherReading.humidity, BoundedNumber)
assert WeatherReading.humidity.storage_name == "_humidity"
assert WeatherReading.temperature.storage_name == "_temperature"
assert not hasattr(reading, "__dict__")
print(reading.station, reading.humidity, reading.temperature)

north 40.0 22.5


### Step 3 — Verify error paths and understand the limit

Data descriptors validate assignments to the public name. A determined caller can still write `_humidity` directly. That is not a bug in the descriptor; it is a reminder that conventional Python attribute access is not a security boundary.

In [25]:
expect_exception(ValueError, lambda: setattr(reading, "humidity", 101))
expect_exception(ValueError, lambda: setattr(reading, "temperature", float("nan")))
expect_exception(TypeError, lambda: setattr(reading, "humidity", True))
expect_exception(AttributeError, lambda: delattr(reading, "humidity"))
assert (reading.humidity, reading.temperature) == (40.0, 22.5)
print("Rejected updates preserve the previous state.")

Rejected updates preserve the previous state.


**Takeaway.** Descriptors reuse *behavior*, and slots prescribe *storage*. Explicit backing slots are what make the combination work.

---

# Problem 09 — Caching a derived value without `__dict__`

**Scenario.** You need an expensive derived property but want to retain a slot-only object.

**Your task.** Demonstrate why `functools.cached_property` is unsuitable without a dictionary, then write an explicit cache slot and invalidate it after mutation.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Why the convenient decorator fails

`functools.cached_property` normally writes a cached result to the instance `__dict__`. A dictionary-free slotted instance does not have that storage. The failure is about the decorator's mechanism, not about ordinary properties.

In [26]:
from functools import cached_property


class IncompatibleCache:
    __slots__ = ("value",)

    def __init__(self, value):
        self.value = value

    @cached_property
    def doubled(self):
        return self.value * 2


cache_error = expect_exception(TypeError, lambda: IncompatibleCache(3).doubled)
print(cache_error)

No '__dict__' attribute on 'IncompatibleCache' instance to cache 'doubled' property.


### Step 2 — Define a dedicated cache slot

The derived value is recomputed only when `_cached_total` contains the sentinel. The setter invalidates the cache; without invalidation, the property can silently serve stale values. Keep the cache implementation close to the mutation API.

In [27]:
CACHE_EMPTY = object()


class InvoiceLine:
    __slots__ = ("_units", "_unit_price", "_cached_total", "compute_count")

    def __init__(self, units, unit_price):
        self._cached_total = CACHE_EMPTY
        self.compute_count = 0
        self.units = units
        self.unit_price = unit_price

    @property
    def units(self):
        return self._units

    @units.setter
    def units(self, value):
        if isinstance(value, bool) or not isinstance(value, int) or value < 0:
            raise ValueError("units must be a nonnegative integer")
        self._units = value
        self._cached_total = CACHE_EMPTY

    @property
    def unit_price(self):
        return self._unit_price

    @unit_price.setter
    def unit_price(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("price must be numeric")
        if not math.isfinite(float(value)) or value < 0:
            raise ValueError("price must be finite and nonnegative")
        self._unit_price = float(value)
        self._cached_total = CACHE_EMPTY

    @property
    def total(self):
        if self._cached_total is CACHE_EMPTY:
            self.compute_count += 1
            self._cached_total = self.units * self.unit_price
        return self._cached_total


invoice = InvoiceLine(3, 19.5)
assert invoice.total == 58.5
assert invoice.total == 58.5
assert invoice.compute_count == 1
invoice.units = 4
assert invoice.total == 78.0
assert invoice.compute_count == 2
assert not hasattr(invoice, "__dict__")
print("Computation count:", invoice.compute_count)

Computation count: 2


### Step 3 — Mutation regression tests

A working cache is not just “compute once.” It must also handle every input change, retain its result when unchanged, and reject bad updates without corrupting a prior value.

In [28]:
invoice.unit_price = 10
assert invoice.total == 40 and invoice.compute_count == 3
assert invoice.total == 40 and invoice.compute_count == 3
expect_exception(ValueError, lambda: setattr(invoice, "units", -2))
expect_exception(ValueError, lambda: setattr(invoice, "unit_price", float("inf")))
assert invoice.total == 40 and invoice.compute_count == 3
print("Cache consistency passed after both valid and invalid mutations.")

Cache consistency passed after both valid and invalid mutations.


**Takeaway.** A cache needs somewhere to live. Reserve a cache slot and a well-defined invalidation path, or deliberately choose dictionary-backed instances if decorator convenience is more important.

---

# Problem 10 — Copying and pickling: preserve the intended state

**Scenario.** You have slotted records containing mutable lists, and they must be copied and serialized.

**Your task.** Compare shallow versus deep copies. Then implement explicit, versioned pickle state and show how to handle a legacy state representation.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Copying is not the same as duplicating nested objects

`copy.copy` creates another outer object but keeps references to nested values; `copy.deepcopy` recursively copies supported nested values. A slot does not automatically make nested data immutable.

In [29]:
class Batch:
    __slots__ = ("name", "samples")

    def __init__(self, name, samples):
        self.name = name
        self.samples = list(samples)


original_batch = Batch("A", [1, 2])
shallow_batch = copy.copy(original_batch)
deep_batch = copy.deepcopy(original_batch)
original_batch.samples.append(3)
assert shallow_batch is not original_batch
assert shallow_batch.samples is original_batch.samples
assert shallow_batch.samples == [1, 2, 3]
assert deep_batch.samples == [1, 2]
print("Shallow:", shallow_batch.samples, "Deep:", deep_batch.samples)

Shallow: [1, 2, 3] Deep: [1, 2]


### Step 2 — Write down a durable serialization contract

Modern Python can often pickle simple slotted objects automatically, but explicit state hooks help with schema evolution. `__getstate__` below emits `(version, mapping)`. `__setstate__` accepts that shape and an older plain dictionary. Pickle is **unsafe for untrusted bytes**; use it only with data you trust.

In [30]:
class VersionedBatch:
    __slots__ = ("name", "samples", "source")

    CURRENT_VERSION = 2

    def __init__(self, name, samples, *, source="manual"):
        self.name = name
        self.samples = list(samples)
        self.source = source

    def __getstate__(self):
        return (self.CURRENT_VERSION, {
            "name": self.name,
            "samples": list(self.samples),
            "source": self.source,
        })

    def __setstate__(self, state):
        if isinstance(state, dict):  # Legacy version 1 had no version tag.
            version, data = 1, state
        else:
            version, data = state
        if version not in (1, 2):
            raise ValueError(f"Unsupported state version: {version}")
        self.name = data["name"]
        self.samples = list(data["samples"])
        self.source = data.get("source", "legacy")


vbatch = VersionedBatch("test", [3, 4], source="sensor")
serialized = pickle.dumps(vbatch)
restored_batch = pickle.loads(serialized)
assert restored_batch is not vbatch
assert restored_batch.samples == [3, 4]
assert restored_batch.source == "sensor"
assert not hasattr(restored_batch, "__dict__")
print("Round-trip bytes:", len(serialized), "; source:", restored_batch.source)

Round-trip bytes: 104 ; source: sensor


### Step 3 — Simulate a legacy state without constructing malicious pickle bytes

Normally `pickle` creates a blank object then calls `__setstate__`. We can model the schema-migration logic directly with `object.__new__`; this is only a **unit test of the state hook**, not a substitute for full round-trip tests.

In [31]:
legacy_batch = object.__new__(VersionedBatch)
legacy_batch.__setstate__({"name": "old", "samples": [9]})
assert legacy_batch.name == "old"
assert legacy_batch.samples == [9]
assert legacy_batch.source == "legacy"
expect_exception(ValueError, lambda: legacy_batch.__setstate__((999, {})))
print("Legacy migration and unsupported-version handling passed.")

Legacy migration and unsupported-version handling passed.


**Takeaway.** Slots do not replace an explicit serialization contract. Keep persistent data formats versioned and never unpickle data supplied by untrusted parties.

---

# Problem 11 — Slotted dataclasses and shallow immutability

**Scenario.** You want automatically generated constructors and equality without typing every special method by hand.

**Your task.** Create a frozen, slotted dataclass, inspect its storage, and demonstrate that `frozen=True` does not recursively freeze mutable fields.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Combine `@dataclass` and slots

Python 3.10 introduced `@dataclass(slots=True)`. The decorator creates a slotted result class. A `frozen=True` dataclass prevents ordinary reassignment of declared fields, but Python is not offering cryptographic or deep immutability.

In [32]:
@dataclass(slots=True, frozen=True)
class FrozenCoordinate:
    latitude: float
    longitude: float


frozen = FrozenCoordinate(19.076, 72.8777)
assert frozen == FrozenCoordinate(19.076, 72.8777)
assert not hasattr(frozen, "__dict__")
print("Field names:", [field.name for field in fields(FrozenCoordinate)])
print("Value:", frozen)
print("Frozen reassignment:", expect_exception(Exception, lambda: setattr(frozen, "latitude", 0.0)))

Field names: ['latitude', 'longitude']
Value: FrozenCoordinate(latitude=19.076, longitude=72.8777)
Frozen reassignment: cannot assign to field 'latitude'


### Step 2 — Probe the exact exception

The generated mutation guard raises `FrozenInstanceError` (a subclass of `AttributeError`) on an attempt to assign a declared field. Test the exact exception rather than accepting every possible failure in production tests.

In [33]:
from dataclasses import FrozenInstanceError

expect_exception(FrozenInstanceError, lambda: setattr(frozen, "latitude", 99.0))
assert frozen.latitude == 19.076

### Step 3 — A frozen outer object can contain a mutable list

If your business requirement is deep immutability, store immutable members such as tuples rather than lists. `slots=True` and `frozen=True` address **different** concerns.

In [34]:
@dataclass(slots=True, frozen=True)
class FrozenOuter:
    labels: list[str]


outer = FrozenOuter(["one"])
outer.labels.append("two")
assert outer.labels == ["one", "two"]
expect_exception(FrozenInstanceError, lambda: setattr(outer, "labels", []))


@dataclass(slots=True, frozen=True)
class ImmutableLabels:
    labels: tuple[str, ...]


immutable = ImmutableLabels(("one", "two"))
assert isinstance(immutable.labels, tuple)
print("Mutable nested list:", outer.labels, "; immutable tuple:", immutable.labels)

Mutable nested list: ['one', 'two'] ; immutable tuple: ('one', 'two')


**Takeaway.** Dataclasses can eliminate boilerplate around slots, but neither slots nor a frozen outer object guarantee that referenced objects are immutable.

---

# Problem 12 — Reflect over slots across a class hierarchy

**Scenario.** A debugging tool must export the currently initialized fields from a family of slotted types.

**Your task.** Build an inheritance-aware inspector that reads slot declarations along the MRO, ignores special slots, and reports uninitialized values without silently inventing defaults.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Why reading only `type(obj).__slots__` fails

A subclass usually declares only *its new slots*. To capture inherited state, inspect the MRO. Also remember: declarations may be strings or iterable collections; a string is **one name**, not a sequence of characters. For this exercise, all slot declarations are conventional strings or tuples, with no conflicting redeclarations.

In [35]:
class AuditBase:
    __slots__ = ("identifier", "status")

    def __init__(self, identifier):
        self.identifier = identifier
        # status is intentionally uninitialized.


class AuditChild(AuditBase):
    __slots__ = "priority"  # A string declaration means one slot.

    def __init__(self, identifier, priority):
        super().__init__(identifier)
        self.priority = priority


audit = AuditChild("audit-1", priority=7)
assert AuditChild.__slots__ == "priority"
print("Only the subclass's declaration:", AuditChild.__slots__)

Only the subclass's declaration: priority


### Step 2 — Normalize declarations and inspect descriptors

A declaration alone can be misleading if a class overrides a name later. For a small diagnostic helper, gather names by walking the MRO, remove `__dict__`/`__weakref__`, and use `hasattr` before reading. In production reflection, also inspect the actual descriptors and account for name mangling or custom attribute access.

In [36]:
def declared_slot_names(cls):
    """Return deduplicated public-ish slot names across a normal class MRO."""
    result = []
    seen = set()
    for base in cls.__mro__:
        declaration = base.__dict__.get("__slots__", ())
        if isinstance(declaration, str):
            declaration = (declaration,)
        for name in declaration:
            if name not in {"__dict__", "__weakref__"} and name not in seen:
                seen.add(name)
                result.append(name)
    return tuple(result)


def snapshot_slots(obj):
    """Map readable initialized slot names to their current values."""
    return {
        name: getattr(obj, name)
        for name in declared_slot_names(type(obj))
        if hasattr(obj, name)
    }


assert declared_slot_names(AuditChild) == ("priority", "identifier", "status")
assert snapshot_slots(audit) == {"priority": 7, "identifier": "audit-1"}
print("Declared:", declared_slot_names(AuditChild))
print("Initialized:", snapshot_slots(audit))

Declared: ('priority', 'identifier', 'status')
Initialized: {'priority': 7, 'identifier': 'audit-1'}


### Step 3 — Add explicit restoration with a narrow contract

Do not blindly write arbitrary state keys into objects. Accept only known names and require callers to opt in to mutation. This helper is intentionally **not** a universal serializer: custom descriptors, read-only attributes, private name mangling, and schema migration need domain-specific handling.

In [37]:
def restore_declared_slots(obj, data):
    valid = set(declared_slot_names(type(obj)))
    unknown = set(data) - valid
    if unknown:
        raise ValueError(f"Unknown slots: {sorted(unknown)}")
    for name, value in data.items():
        setattr(obj, name, value)


restored_audit = object.__new__(AuditChild)
restore_declared_slots(restored_audit, snapshot_slots(audit))
assert restored_audit.identifier == "audit-1"
assert restored_audit.priority == 7
assert not hasattr(restored_audit, "status")
expect_exception(ValueError, lambda: restore_declared_slots(restored_audit, {"typo": 10}))
print("Known fields restored; unknown fields rejected.")

Known fields restored; unknown fields rejected.


**Takeaway.** Reflection needs to account for inherited declarations and uninitialized slots. An inspector is not automatically a safe or general-purpose serializer.

---

# Problem 13 — Allow one extension pocket intentionally

**Scenario.** An application wants fixed core state while still permitting plugin metadata.

**Your task.** Use both ordinary slots and `__dict__` explicitly, distinguish their storage locations, and decide whether the hybrid tradeoff is worthwhile.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Mixed storage is allowed by design

Including `"__dict__"` in slots requests a dictionary *in addition* to the named slot storage. This sacrifices the no-dictionary memory advantage, but can be useful for integration points or dynamically attached annotations.

In [38]:
class ExtensibleEvent:
    __slots__ = ("event_id", "timestamp", "__dict__")

    def __init__(self, event_id, timestamp):
        self.event_id = event_id
        self.timestamp = timestamp


event = ExtensibleEvent("evt-9", 123456)
event.plugin_label = "dashboard"
event.validation_status = "reviewed"
assert event.event_id == "evt-9"
assert event.timestamp == 123456
assert event.__dict__ == {
    "plugin_label": "dashboard",
    "validation_status": "reviewed",
}
assert "event_id" not in event.__dict__
print("Dictionary-backed extension metadata:", event.__dict__)

Dictionary-backed extension metadata: {'plugin_label': 'dashboard', 'validation_status': 'reviewed'}


### Step 2 — Decide whether hybrid storage fits the need

If all you need is an extensible collection of metadata, a *single explicit metadata slot* may be clearer than allowing arbitrary attributes. The latter makes typo detection harder; the former can validate names and reserve the core attribute namespace.

In [39]:
class ExplicitMetadataEvent:
    __slots__ = ("event_id", "metadata")

    def __init__(self, event_id):
        self.event_id = event_id
        self.metadata = {}


explicit_event = ExplicitMetadataEvent("evt-10")
explicit_event.metadata["plugin_label"] = "dashboard"
expect_exception(AttributeError, lambda: setattr(explicit_event, "plugin_lable", "typo"))
assert not hasattr(explicit_event, "__dict__")
assert explicit_event.metadata["plugin_label"] == "dashboard"
print("Explicit extension namespace:", explicit_event.metadata)

Explicit extension namespace: {'plugin_label': 'dashboard'}


**Takeaway.** You can opt back into a dictionary explicitly. Decide whether plugins should add *attributes* or only entries in an explicitly managed metadata mapping.

---

# Problem 14 — Measure memory honestly

**Scenario.** A team wants evidence that slots help when it instantiates many small objects.

**Your task.** Compare equivalent regular and slotted classes using both `sys.getsizeof` and `tracemalloc`. Avoid claiming that one measurement proves universal memory or speed gains.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Build comparable objects

Use the same values and constructor logic. `sys.getsizeof(instance)` measures the object's *shallow* size; for a dictionary-backed instance, the separate instance dictionary is not included in that number. For the slotted object, the inline slot space usually contributes to its shallow size.

In [40]:
class DictionaryParticle:
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z


class SlottedParticle:
    __slots__ = ("x", "y", "z")

    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z


dict_particle = DictionaryParticle(1, 2, 3)
slot_particle = SlottedParticle(1, 2, 3)
print("Dict instance shallow:", sys.getsizeof(dict_particle))
print("Dict's extra mapping shallow:", sys.getsizeof(dict_particle.__dict__))
print("Slotted instance shallow:", sys.getsizeof(slot_particle))
assert (dict_particle.x, dict_particle.y, dict_particle.z) == (slot_particle.x, slot_particle.y, slot_particle.z)

Dict instance shallow: 48
Dict's extra mapping shallow: 296
Slotted instance shallow: 56


### Step 2 — Measure a complete allocation batch

`tracemalloc` measures Python-tracked allocation changes while we retain a list of live objects. Create one batch at a time, and **stop/restart tracing** between alternatives. These numbers include list storage and other tracked allocations: they are useful relative evidence in the current process, not a portable accounting of every byte of process RAM.

In [41]:
def tracked_batch_peak(factory, count=12_000):
    gc.collect()
    tracemalloc.start()
    try:
        objects = [factory(i, i + 1, i + 2) for i in range(count)]
        current, peak = tracemalloc.get_traced_memory()
        assert len(objects) == count
        return current, peak
    finally:
        tracemalloc.stop()


dict_current, dict_peak = tracked_batch_peak(DictionaryParticle)
slot_current, slot_peak = tracked_batch_peak(SlottedParticle)
print(f"Dictionary-backed: current={dict_current:,} B; peak={dict_peak:,} B")
print(f"Slotted:           current={slot_current:,} B; peak={slot_peak:,} B")
print("Measured current difference (dict - slot):", dict_current - slot_current, "B")
assert dict_current > 0 and slot_current > 0

Dictionary-backed: current=2,391,169 B; peak=2,391,201 B
Slotted:           current=1,907,384 B; peak=1,907,416 B
Measured current difference (dict - slot): 483785 B


### Step 3 — Probe speed, but do not turn it into a universal claim

Attribute lookup speed can vary with CPython version, CPU, cache effects, object layout and benchmark design. A microbenchmark is neither an end-to-end workload nor a guarantee that slots are faster. Record measurements; decide based on representative profiling.

In [42]:
def read_xyz(obj):
    return obj.x + obj.y + obj.z


dict_seconds = min(timeit.repeat(lambda: read_xyz(dict_particle), number=100_000, repeat=3))
slot_seconds = min(timeit.repeat(lambda: read_xyz(slot_particle), number=100_000, repeat=3))
print(f"Read 100,000 times (best of 3): dict={dict_seconds:.6f}s; slot={slot_seconds:.6f}s")
assert read_xyz(dict_particle) == read_xyz(slot_particle) == 6

Read 100,000 times (best of 3): dict=0.008316s; slot=0.010970s


### Step 4 — Interpret the result

A stronger real-world comparison also examines resident set size, garbage collection, allocation patterns, extra descriptors, inheritance and developer complexity. Choose slots for a documented need (often many fixed-shape instances), not simply because the benchmark printed a smaller number once.

**Takeaway.** Shallow sizes, traced allocation totals, and runtime timings answer different questions. Never equate one with an unconditional “performance boost.”

---

# Problem 15 — Capstone: a bounded, cache-aware sensor reading

**Scenario.** An ingest service stores many readings, validates inputs, calculates a derived alert status, exposes weak references for an index, and exports a safe plain-data representation.

**Your task.** Combine the prior ideas incrementally into a production-oriented class. Then run lifecycle, validation, cache, copy, weak-reference, and bulk smoke tests.

**How to study it.** Read the prediction question, run the small experiment, then examine the worked solution and its assertions. Feel free to modify the code and rerun the notebook from the top.

### Step 1 — Translate requirements into storage

We need slots for `sensor_id`, backing values `_temperature`/`_humidity`, and cache `_alert_cache`. The `__weakref__` slot lets external indexes refer to readings without extending their lifetime. We deliberately do not include `__dict__`.

The alert policy for this *exercise* is not a weather standard: `True` when temperature is strictly above 35 °C or humidity is strictly above 90%. Define it explicitly so the boundary tests can be precise.

### Step 2 — Decide validation and mutation policy

Reuse `BoundedNumber` from Problem 8, but wrap its setters in public properties? A descriptor cannot easily invalidate an independent cache without an explicit mechanism. Here we choose straightforward properties, and a shared coordinate-style validation helper is **not** used because accepted temperature/humidity ranges differ. `sensor_id` is a string that is fixed after initialization by convention; this is not a security boundary.

In [43]:
def check_sensor_number(value, name, minimum, maximum):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(f"{name} must be an int or float, not bool")
    number = float(value)
    if not math.isfinite(number) or not minimum <= number <= maximum:
        raise ValueError(f"{name} must be finite and in [{minimum}, {maximum}]")
    return number


assert check_sensor_number(0, "humidity", 0, 100) == 0.0
expect_exception(ValueError, lambda: check_sensor_number(101, "humidity", 0, 100))

'humidity must be finite and in [0, 100]'

### Step 3 — Full worked solution

Initializing the cache before setting validated properties keeps setters safe during construction. The derived property caches a boolean; any successful input update invalidates it. `to_dict` exports only stable plain data, not the cache or weak-reference machinery.

In [44]:
class SensorReading:
    __slots__ = (
        "sensor_id", "_temperature", "_humidity", "_alert_cache", "__weakref__"
    )

    def __init__(self, sensor_id, *, temperature, humidity):
        if not isinstance(sensor_id, str) or not sensor_id.strip():
            raise ValueError("sensor_id must be a nonempty string")
        self.sensor_id = sensor_id
        self._alert_cache = CACHE_EMPTY
        self.temperature = temperature
        self.humidity = humidity

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, value):
        validated = check_sensor_number(value, "temperature", -100, 80)
        self._temperature = validated
        self._alert_cache = CACHE_EMPTY

    @property
    def humidity(self):
        return self._humidity

    @humidity.setter
    def humidity(self, value):
        validated = check_sensor_number(value, "humidity", 0, 100)
        self._humidity = validated
        self._alert_cache = CACHE_EMPTY

    @property
    def alert(self):
        if self._alert_cache is CACHE_EMPTY:
            self._alert_cache = self.temperature > 35 or self.humidity > 90
        return self._alert_cache

    def to_dict(self):
        return {
            "sensor_id": self.sensor_id,
            "temperature": self.temperature,
            "humidity": self.humidity,
        }

    @classmethod
    def from_dict(cls, data):
        expected = {"sensor_id", "temperature", "humidity"}
        if set(data) != expected:
            raise ValueError(f"Expected exactly these keys: {sorted(expected)}")
        return cls(
            data["sensor_id"],
            temperature=data["temperature"],
            humidity=data["humidity"],
        )

    def __repr__(self):
        return (
            f"SensorReading({self.sensor_id!r}, temperature={self.temperature!r}, "
            f"humidity={self.humidity!r})"
        )


sensor = SensorReading("S-007", temperature=23, humidity=40)
print(sensor)
assert sensor.alert is False
assert not hasattr(sensor, "__dict__")

SensorReading('S-007', temperature=23.0, humidity=40.0)


### Step 4 — Boundaries, invalid mutations and cache invalidation

Strict `>` comparisons mean exactly 35 °C and exactly 90% humidity do **not** trigger an alert. The setter validates first and mutates second; rejected updates must leave the existing state unchanged.

In [45]:
sensor.temperature = 35
sensor.humidity = 90
assert sensor.alert is False
sensor.temperature = 35.1
assert sensor.alert is True
sensor.temperature = 20
assert sensor.alert is False
sensor.humidity = 90.1
assert sensor.alert is True
prior = sensor.to_dict()
expect_exception(ValueError, lambda: setattr(sensor, "humidity", 101))
expect_exception(ValueError, lambda: setattr(sensor, "temperature", float("nan")))
expect_exception(TypeError, lambda: setattr(sensor, "temperature", True))
assert sensor.to_dict() == prior
assert sensor.alert is True
expect_exception(AttributeError, lambda: setattr(sensor, "debug", True))
print("Boundary, mutation, and unknown-field tests passed.")

Boundary, mutation, and unknown-field tests passed.


### Step 5 — A plain-data round trip

The explicit dictionary representation is suitable for later JSON encoding (all its values here are JSON-compatible primitives). Unlike pickle, it does not execute arbitrary object-reconstruction instructions. It still needs schema validation, as the constructor and `from_dict` provide here.

In [46]:
import json

wire_data = json.dumps(sensor.to_dict(), sort_keys=True)
rebuilt_sensor = SensorReading.from_dict(json.loads(wire_data))
assert rebuilt_sensor.to_dict() == sensor.to_dict()
assert rebuilt_sensor is not sensor
assert rebuilt_sensor.alert == sensor.alert
expect_exception(ValueError, lambda: SensorReading.from_dict({"sensor_id": "bad"}))
expect_exception(ValueError, lambda: SensorReading.from_dict({**sensor.to_dict(), "unknown": 1}))
print("JSON round trip:", wire_data)

JSON round trip: {"humidity": 90.1, "sensor_id": "S-007", "temperature": 20.0}


### Step 6 — Weak references and scale

A separate weak-reference index should not keep readings alive. For a bulk smoke test, create many valid objects and verify one boundary condition across them. This is a **correctness** check, not a memory benchmark.

In [47]:
sensor_ref = weakref.ref(sensor)
assert sensor_ref() is sensor
bulk = [
    SensorReading(f"S-{i:05d}", temperature=(i % 60) - 10, humidity=i % 101)
    for i in range(2_500)
]
assert all(not hasattr(obj, "__dict__") for obj in bulk)
assert sum(obj.alert for obj in bulk) == sum(
    obj.temperature > 35 or obj.humidity > 90 for obj in bulk
)
assert len({obj.sensor_id for obj in bulk}) == len(bulk)
print("Bulk smoke test passed:", len(bulk), "unique dictionary-free readings.")

Bulk smoke test passed: 2500 unique dictionary-free readings.


### Step 7 — Review tradeoffs

- **Benefits:** predictable core layout, typo rejection for new instance names, central validation, explicit cache invalidation, opt-in weak references.
- **Costs:** more handwritten code, less spontaneous extensibility, extra care when adding future fields, and direct backing-attribute access is still possible.
- **Design alternatives:** `@dataclass(slots=True)` for less boilerplate; an ordinary dictionary-backed class when extension flexibility matters more; immutable tuples for truly immutable nested state; explicit JSON schema/versioning for persistent storage.

The correct choice depends on an observed workload and the public API contract, not a rule that every class should use slots.

---

# Final mastery challenges (optional, deliberately not pre-solved in code)

You have just studied 15 fully worked advanced problems. To check independent understanding, try these follow-ups **without looking back**. They extend the solved cases rather than requiring outside libraries.

1. Modify `SensorReading` so that a successful assignment of the *same* temperature does not invalidate the cache; explain whether this extra complexity is justified.
2. Write `from_dict` for a future version containing an optional `location`, and preserve backwards compatibility without allowing arbitrary unknown keys.
3. Implement a thread-safety strategy for a mutable cached reading. Explain why a slot alone cannot make multi-step state changes atomic.
4. Modify `snapshot_slots` to handle double-underscore slot declarations via owner-dependent name mangling and descriptor checks; explain how `__getattribute__` overrides complicate the inspector.
5. Re-run the allocation benchmark on two Python implementations or versions, and write down why the measured difference might change.

**Best-practice checklist:** understand inheritance; avoid slot/property name collisions; never rely on slots for privacy or immutability; decide on weak references and dictionaries when designing the class; profile realistic workloads; keep serialization explicit when schema changes matter.

## Notebook completion check

If **Restart Kernel → Run All** reaches this cell with no exception, every implemented exercise and regression assertion executed successfully. Re-run earlier cells after changing any implementation to catch cross-exercise regressions.

In [48]:
assert mumbai.longitude == 72.8777
assert snapshot_slots(audit)["priority"] == 7
assert rebuilt_sensor.to_dict() == sensor.to_dict()
assert len(bulk) == 2_500
print("SUCCESS: all 15 tutorial problems completed and checked.")

SUCCESS: all 15 tutorial problems completed and checked.
